# LAB | Ensemble Methods

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In this Lab, you should try different ensemble methods in order to see if can obtain a better model than before. In order to do a fair comparison, you should perform the same feature scaling, engineering applied in previous Lab.

In [1]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [3]:
spaceship = spaceship.dropna()

In [4]:
from sklearn.model_selection import train_test_split

X = spaceship.drop("Transported", axis=1)
y = spaceship["Transported"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [5]:
from sklearn.preprocessing import OneHotEncoder

onehot_columns = ["Cabin", "HomePlanet", "Destination"]
drop_columns = ["PassengerId", "Name"]
bool_columns = ["CryoSleep", "VIP"]

X_train = X_train.copy()
X_test = X_test.copy()

X_train["Cabin"] = X_train["Cabin"].str[0]
X_test["Cabin"] = X_test["Cabin"].str[0]

onehot_encoder = OneHotEncoder(
    sparse_output=False,
    handle_unknown="ignore"
)

encoded_train = onehot_encoder.fit_transform(X_train[onehot_columns])
encoded_test = onehot_encoder.transform(X_test[onehot_columns])

encoded_train_df = pd.DataFrame(
    encoded_train,
    columns=onehot_encoder.get_feature_names_out(onehot_columns),
    index=X_train.index
)

encoded_test_df = pd.DataFrame(
    encoded_test,
    columns=onehot_encoder.get_feature_names_out(onehot_columns),
    index=X_test.index
)

X_train = pd.concat(
    [
        X_train.drop(columns=onehot_columns + drop_columns),
        encoded_train_df
    ],
    axis=1
)

X_test = pd.concat(
    [
        X_test.drop(columns=onehot_columns + drop_columns),
        encoded_test_df
    ],
    axis=1
)

X_train[bool_columns] = X_train[bool_columns].astype(int)
X_test[bool_columns] = X_test[bool_columns].astype(int)

X_test = X_test.reindex(columns=X_train.columns, fill_value=0)


In [6]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

**Perform Train Test Split**

In [7]:
#your code here

**Model Selection** - now you will try to apply different ensemble methods in order to get a better model

- Bagging и Pasting

In [8]:
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier

bag_clf = BaggingClassifier(
    estimator=DecisionTreeClassifier(random_state=42),
    n_estimators=100,
    bootstrap=True,
    random_state=42
)

bag_clf.fit(X_train, y_train)
y_pred_bag = bag_clf.predict(X_test)

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Accuracy:")
print(accuracy_score(y_test, y_pred_bag))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_bag))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_bag))

Accuracy:
0.8048411497730711

Classification Report:
              precision    recall  f1-score   support

       False       0.80      0.81      0.80       653
        True       0.81      0.80      0.81       669

    accuracy                           0.80      1322
   macro avg       0.80      0.80      0.80      1322
weighted avg       0.80      0.80      0.80      1322


Confusion Matrix:
[[529 124]
 [134 535]]


- Random Forests

In [15]:
from sklearn.ensemble import RandomForestClassifier

rf_clf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_clf.fit(X_train, y_train)
y_pred_rf = rf_clf.predict(X_test)

print(accuracy_score(y_test, y_pred_rf))

0.8048411497730711


In [16]:

print(classification_report(y_test, y_pred_rf))

              precision    recall  f1-score   support

       False       0.79      0.82      0.81       653
        True       0.82      0.79      0.80       669

    accuracy                           0.80      1322
   macro avg       0.81      0.81      0.80      1322
weighted avg       0.81      0.80      0.80      1322



- Gradient Boosting

In [17]:
from sklearn.ensemble import GradientBoostingClassifier

gb_clf = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

gb_clf.fit(X_train, y_train)
y_pred_gb = gb_clf.predict(X_test)

print("Accuracy:")
print(accuracy_score(y_test, y_pred_gb))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_gb))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_gb))

Accuracy:
0.8101361573373677

Classification Report:
              precision    recall  f1-score   support

       False       0.84      0.76      0.80       653
        True       0.78      0.86      0.82       669

    accuracy                           0.81      1322
   macro avg       0.81      0.81      0.81      1322
weighted avg       0.81      0.81      0.81      1322


Confusion Matrix:
[[495 158]
 [ 93 576]]


- Adaptive Boosting

In [18]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

ada_clf = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1, random_state=42),
    n_estimators=100,
    learning_rate=0.1,
    random_state=42
)

ada_clf.fit(X_train, y_train)
y_pred_ada = ada_clf.predict(X_test)

print("Accuracy:")
print(accuracy_score(y_test, y_pred_ada))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_ada))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_ada))

Accuracy:
0.7450832072617246

Classification Report:
              precision    recall  f1-score   support

       False       0.71      0.83      0.76       653
        True       0.80      0.66      0.72       669

    accuracy                           0.75      1322
   macro avg       0.75      0.75      0.74      1322
weighted avg       0.75      0.75      0.74      1322


Confusion Matrix:
[[543 110]
 [227 442]]


Which model is the best and why?

In [ ]:
Gradient Boosting has best F1-score and accuracy. It is a powerful ensemble method that builds trees sequentially, where each tree tries to correct the errors of the previous one. This often leads to better performance compared to other ensemble methods like Bagging and Random Forest, which build trees independently. AdaBoost can also perform well, but it may be more sensitive to noisy data and outliers compared to Gradient Boosting.